# Notebook 02 - Lam sach du lieu Ha Noi

Muc tieu:
- Doc du lieu tho da enrich truong sau.
- Chon pham vi linh hoat: 1 quan hoac top-N quan de du so mau train.
- Lam sach missing, loai outlier, tao tap du lieu san sang train.
- Luu ra `data/processed/housing_hanoi_clean.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "housing_raw_vn_hanoi_deep.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH

WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/raw/housing_raw_vn_hanoi_deep.csv')

In [2]:
df = pd.read_csv(RAW_PATH)
print("Shape ban dau:", df.shape)
df.head(3)

Shape ban dau: (600, 28)


,source,city,title,price_raw,area_raw,location_raw,detail_url,list_id,category_name,crawl_page,...,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,DiaChi,moTa,latitude,longitude
0,chotot,Ha Noi,Chính chủ nhờ bán căn nhà phố Lĩnh Nam - Hoàng...,"5,2 tỷ",36.0,"Phường Lĩnh Nam, Ha Noi",https://cdn.chotot.com/8UBB8MTaVWgAalI0qsjJMOB...,132039968,Nhà ở,1,...,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Lĩnh Nam,Quận Hoàng Mai,"Đường Lĩnh Nam, Phường Lĩnh Nam, Quận Hoàng Ma...",Chính chủ nhờ bán căn nhà phố Lĩnh Nam - Hoàng...,20.979588,105.88912
1,chotot,Ha Noi,"Nhà mặt phố Văn Chương, mặt hồ, kinh doanh, 3 ...",18 tỷ,65.0,"Phường Văn Chương, Ha Noi",https://cdn.chotot.com/nuGKzP03tDa_PDr0sEhgtYa...,132039918,Nhà ở,1,...,NaN,"Nhà mặt phố, mặt tiền",NaN,NaN,Phường Văn Chương,Quận Đống Đa,"Phố Hồ Văn Chương, Phường Văn Chương, Quận Đốn...","Chủ nhà cần bán nhà mặt phố Văn Chương, ngay m...",21.021517,105.83358
2,chotot,Ha Noi,"Đại Mỗ - Vạn Phúc: 54m2, 5 tầng mới, 3PN, sổ đ...","7,15 tỷ",35.0,"Phường Đại Mỗ, Ha Noi",https://cdn.chotot.com/-9CJNrOK5zyUX0vlJRKW7Hw...,131581480,Nhà ở,1,...,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Đại Mỗ,Quận Nam Từ Liêm,"Phố Ngọc Trục, Phường Đại Mỗ, Quận Nam Từ Liêm...",Gia đình cần bán gấp căn nhà tại phố Ngọc Trục...,20.988243,105.77060


## 1) Chon 1 quan theo so luong mau

In [3]:
district_counts = df["Quan"].fillna("Khong ro").value_counts()
district_counts.head(15)

Quan
Quận Long Biên       94
Quận Hai Bà Trưng    85
Quận Hoàng Mai       81
Quận Đống Đa         58
Quận Cầu Giấy        44
Quận Hà Đông         43
Quận Thanh Xuân      38
Quận Nam Từ Liêm     24
Quận Tây Hồ          24
Quận Bắc Từ Liêm     23
Quận Ba Đình         22
Huyện Thanh Trì      16
Huyện Hoài Đức       12
Huyện Quốc Oai        9
Huyện Đông Anh        8
Name: count, dtype: int64

In [4]:
# Luon chon duy nhat 1 quan co nhieu mau nhat (bo qua gia tri khong ro)
valid_district_counts = district_counts[district_counts.index != "Khong ro"]
TARGET_DISTRICT = valid_district_counts.index[0]
selected_districts = [TARGET_DISTRICT]

df1 = df[df["Quan"].isin(selected_districts)].copy()
print("Quan duoc chon:", TARGET_DISTRICT)
print("So mau trong quan:", len(df1))

Quan duoc chon: Quận Long Biên
So mau trong quan: 94


## 2) Chuan hoa kieu du lieu

In [5]:
num_cols = [
    "Gia", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang", "latitude", "longitude"
]

for c in num_cols:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

df1[num_cols].dtypes

Gia             int64
dienTich      float64
chieuNgang    float64
chieuDai      float64
Phongngu        int64
PhongTam      float64
SoTang        float64
latitude      float64
longitude     float64
dtype: object

## 3) Xu ly missing co ban

In [6]:
print("Missing truoc:\n", df1[num_cols + ["Loai", "GiayTo", "TinhTrangNoiThat"]].isna().sum())

Missing truoc:
 Gia                  0
dienTich             0
chieuNgang          46
chieuDai            64
Phongngu             0
PhongTam            32
SoTang              25
latitude             0
longitude            0
Loai                 0
GiayTo              20
TinhTrangNoiThat    50
dtype: int64


In [7]:
# Cac cot bat buoc toi thieu cho bai toan gia nha
df1 = df1.dropna(subset=["Gia", "dienTich"])

# Dien median cho cot so (giu lai nhieu mau hon)
for c in ["Phongngu", "PhongTam", "SoTang", "chieuNgang", "chieuDai"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna(df1[c].median())

# Dien nhan 'Khong ro' cho cot phan loai
for c in ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna("Khong ro")

print("Shape sau xu ly missing:", df1.shape)

Shape sau xu ly missing: (94, 28)


## 4) Loai outlier co ban

In [8]:
# Dieu kien hop ly so bo (noi rong de khong loai qua nhieu mau)
cond = (
    (df1["Gia"] >= 2e8) & (df1["Gia"] <= 5e11) &
    (df1["dienTich"] >= 8) & (df1["dienTich"] <= 1500) &
    (df1["Phongngu"] >= 0) & (df1["Phongngu"] <= 30) &
    (df1["PhongTam"] >= 0) & (df1["PhongTam"] <= 30) &
    (df1["SoTang"] >= 0) & (df1["SoTang"] <= 50)
)
df2 = df1[cond].copy()

df2["Gia_m2"] = df2["Gia"] / df2["dienTich"]
print("Shape sau loc outlier:", df2.shape)
df2[["Gia", "dienTich", "Gia_m2"]].describe()

Shape sau loc outlier: (93, 29)


,Gia,dienTich,Gia_m2
count,9.300000e+01,93.000000,9.300000e+01
mean,1.414406e+10,60.006452,2.243748e+08
std,1.328975e+10,35.675829,7.283344e+07
min,4.300000e+09,20.000000,6.555556e+07
25%,7.600000e+09,40.000000,1.750000e+08
50%,1.050000e+10,52.000000,2.142857e+08
75%,1.450000e+10,65.000000,2.709091e+08
max,9.000000e+10,250.000000,5.060976e+08


## 5) Chon cot dau vao train

In [9]:
final_cols = [
    "Gia", "Gia_m2", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang",
    "Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan",
    "latitude", "longitude", "title", "moTa", "list_id"
]

final_cols = [c for c in final_cols if c in df2.columns]
df_clean = df2[final_cols].drop_duplicates(subset=["list_id"]).copy()
print("Shape cuoi:", df_clean.shape)
df_clean.head(3)

Shape cuoi: (93, 18)


,Gia,Gia_m2,dienTich,chieuNgang,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,latitude,longitude,title,moTa,list_id
7,16000000000,3.018868e+08,53.0,4.5000,10.0,3,5.0,6.0,"Nhà ngõ, hẻm",Khong ro,Khong ro,Phường Sài Đồng,Quận Long Biên,21.034845,105.91271,📢📢📢 GẦN NGAY MẶT PHỐ SÀI ĐỒNG - LONG BIÊN | 53...,"✨️Diện tích 53m2 / 6 tầng\n- Mặt tiền 4.5m, hư...",132039613
11,27000000000,2.934783e+08,92.0,5.0000,18.0,6,6.0,8.0,"Nhà ngõ, hẻm",Đã có sổ,Nội thất cao cấp,Phường Gia Thụy,Quận Long Biên,21.048788,105.88481,Bán Gấp phân lô quân đội 92m2 8 tầng thang máy...,BÁN GẤP NHÀ KHU VIP PHÂN LÔ QUÂN ĐỘI 918 – PHÚ...,131995974
13,6600000000,2.062500e+08,32.0,4.6999,10.0,3,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Khong ro,Phường Việt Hưng,Quận Long Biên,21.056430,105.90300,Chính chủ cần bán phố Kim Quan 32m2- 5T- 3 ph...,Chính chủ cần bán phố Kim Quan 32m2- 5T- 3 ph...,129972265


In [10]:
out_path = PROCESSED_DIR / "housing_hanoi_clean.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Da luu du lieu clean tai: {out_path}")

Da luu du lieu clean tai: C:\Users\asus\Desktop\DuDoanGiaNha\data\processed\housing_hanoi_clean.csv


## Mau bao cao (Notebook 02)

- Du lieu dau vao: `data/raw/housing_raw_vn_hanoi_deep.csv`.
- Chien luoc pham vi: gioi han theo 1 quan de giam do lech dia ly.
- Xu ly missing: loai bo mau thieu cot bat buoc (`Gia`, `dienTich`, `Phongngu`), bo sung median cho bien so, gan nhan `Khong ro` cho bien phan loai.
- Xu ly ngoai lai: loc theo mien gia tri hop ly cua `Gia`, `dienTich`, `Phongngu`, `PhongTam`, `SoTang`.
- Du lieu dau ra: `data/processed/housing_hanoi_clean.csv`, san sang cho buoc huan luyen mo hinh.